# M07 Circuit Tracing Tools


## 看图背后的工作流

加载 tracing 工作流，并把它和共享 attribution artifact 里的边分数分布对照起来。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(candidate)], check=True)
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import json
import matplotlib.pyplot as plt

workflow = json.loads((root / "artifacts" / "m07_tracing_tool_workflow.json").read_text())
graph = json.loads((root / "artifacts" / "m06_attribution_graph.json").read_text())
edge_scores = [edge["score"] for case in graph["cases"] for edge in case["edges"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(
    [step["title_en"] for step in workflow["steps"]],
    [step["estimated_minutes"] for step in workflow["steps"]],
    color="#1f5f8b",
)
axes[0].set_title("Workflow stages")
axes[0].tick_params(axis="x", rotation=24)

axes[1].hist(edge_scores, bins=6, color="#c96a28", edgecolor="white")
axes[1].set_title("Edge score distribution")
axes[1].set_xlabel("score")
plt.tight_layout()

print("Workflow outputs:")
for step in workflow["steps"]:
    print("-", step["title_en"], "->", step["output"], f"({step['estimated_minutes']} min)")


## 小结

工具层会决定你能得到什么 trace、它们有多贵，以及最后到底好不好读。


## 把这本 notebook 变成研究产出

研究工作单 means this notebook is not complete when the cells finish. 请配合 /templates 里的模板，把结果写成可复查的输出。


### 运行前

- 在加载 artifact 之前，先写下你预期的 workflow 阶段。
- 提前挑一个你觉得成本或延迟会占主导的位置。
- 先想清楚研究工程师真正需要工具输出什么。


### 运行后

- 指出最会改变研究问题形状的那个 bottleneck。
- 写一个具体的工具改进请求。


### 最后交付这些产物

- 1 张 workflow 图。
- 1 份 bottleneck analysis。
- 1 份可以直接交给工程师的 tooling-needs note。


## 验收题

- 在 tracing workflow 里，哪一个步骤最像瓶颈，为什么？
- 如果你要给研究工程师提一个优先实现的工具需求，你会提什么，怎么衡量它值不值得做？
- 为什么说工具层不仅影响效率，还会改变团队能问什么研究问题？
- 如果你不能在不重看讲义的情况下独立答出其中至少 2 题，就回去重看文章和你的书面输出。
